

-sunset/rise & tide?
-moonset/rise & tide?


In [3]:
import pandas as pd

df_weather = pd.read_csv('../data/processed/coop_ndbc_joined_with_astronomy.csv')
df_survey = pd.read_csv('../data/processed/survey_data_processed.csv')

df_weather.head()

,timestamp,water_level,wind_speed_ms,wind_gust_ms,air_temp_c,water_temp_c,dewpoint_c,wind_direction_deg,air_pressure_hpa,sunrise_time,sunset_time,moonrise_time,moonset_time,moon_phase_age_days,sun_distance_km,moon_distance_km
0,2014-02-01 01:00:00,0.023,5.30,6.35,3.90,4.55,-1.9,281.0,1022.05,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474073e+08,361404.808532
1,2014-02-01 02:00:00,1.245,5.50,6.35,3.60,4.55,-4.4,283.0,1022.70,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474081e+08,362653.467797
2,2014-02-01 03:00:00,2.182,4.25,5.15,3.50,4.50,-4.7,269.5,1023.20,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474090e+08,363762.673757
3,2014-02-01 04:00:00,2.384,4.55,5.55,3.75,4.45,-5.1,279.5,1023.85,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474098e+08,364671.158659
4,2014-02-01 05:00:00,2.370,4.30,4.85,3.55,4.40,-5.1,262.0,1024.05,2014-02-01 07:05:43,2014-02-01 17:14:34,2014-02-01 07:54:00,NaN,1.666778,1.474107e+08,365330.514787


In [6]:
import numpy as np
import pandas as pd

# Assumptions used in this feature block:
assumptions = [
    "Trip timestamp marks trip end time in df_survey.",
    "Trip window is (end_time - hrsf, end_time], inclusive of end.",
    "Weather rows may have time gaps; compare available consecutive rows as-is.",
    "If a trip has no in-window rows OR metric values are all missing, use nearest weather row for averages.",
    "If rise/fall cannot be inferred (fewer than 2 wind-speed values), set flags to 0.",
]

def pick_timestamp_column(df: pd.DataFrame, candidates: list[str], df_name: str) -> str:
    """Pick a timestamp column with explicit candidates first, then sensible fallbacks."""
    for col in candidates:
        if col in df.columns:
            return col

    datetime_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
    if len(datetime_cols) == 1:
        return datetime_cols[0]

    name_fallbacks = [
        c for c in df.columns
        if any(token in c.lower() for token in ["time", "date", "timestamp", "datetime"])
    ]
    if len(name_fallbacks) == 1:
        return name_fallbacks[0]

    raise ValueError(
        f"Could not uniquely determine timestamp column for {df_name}. "
        f"Please set it explicitly. Candidate-like columns: {name_fallbacks}"
    )

def pick_column(df: pd.DataFrame, candidates: list[str], df_name: str, purpose: str) -> str:
    """Pick a column by exact then case-insensitive match."""
    for col in candidates:
        if col in df.columns:
            return col

    lowered_map = {c.lower(): c for c in df.columns}
    for col in candidates:
        if col.lower() in lowered_map:
            return lowered_map[col.lower()]

    raise ValueError(
        f"Could not find {purpose} column in {df_name}. Tried: {candidates}. "
        f"Available columns include: {list(df.columns)[:25]}"
    )

# Keep originals untouched for QA check
df_survey_before = df_survey.copy(deep=True)

# Work on local copies so df_survey/df_weather are not mutated
survey_work = df_survey.copy(deep=True)
weather_work = df_weather.copy(deep=True)

# 1) Preprocess and enforce assumptions
survey_time_col = pick_timestamp_column(
    survey_work,
    candidates=["SURVEY_TIMESTAMP", "survey_timestamp", "timestamp", "trip_end_time"],
    df_name="df_survey",
)
weather_time_col = pick_timestamp_column(
    weather_work,
    candidates=["DATETIME", "datetime", "timestamp", "weather_timestamp", "date_time", "time"],
    df_name="df_weather",
)

hrsf_col = pick_column(
    survey_work,
    candidates=["hrsf", "HRSF", "hours_fished", "trip_hours"],
    df_name="df_survey",
    purpose="hours-fished",
)

wind_speed_col = pick_column(
    weather_work,
    candidates=["wind_speed_ms", "WIND_SPEED_MS", "wind_speed"],
    df_name="df_weather",
    purpose="wind speed",
)
wind_gust_col = pick_column(
    weather_work,
    candidates=["wind_gust_ms", "WIND_GUST_MS", "wind_gust"],
    df_name="df_weather",
    purpose="wind gust",
)

survey_work[survey_time_col] = pd.to_datetime(survey_work[survey_time_col], errors="coerce")
weather_work[weather_time_col] = pd.to_datetime(weather_work[weather_time_col], errors="coerce")
survey_work[hrsf_col] = pd.to_numeric(survey_work[hrsf_col], errors="coerce")

weather_work = weather_work.sort_values(weather_time_col).reset_index(drop=True)

# 2) Create output dataframe
df_tripdata = survey_work.copy(deep=True)
df_tripdata["avg_wind_speed"] = 0.0
df_tripdata["avg_wind_gust"] = 0.0
df_tripdata["wind_speed_rose"] = 0
df_tripdata["wind_speed_fell"] = 0

weather_times = weather_work[weather_time_col]
weather_speed = pd.to_numeric(weather_work[wind_speed_col], errors="coerce")
weather_gust = pd.to_numeric(weather_work[wind_gust_col], errors="coerce")

empty_window_count = 0
fallback_trip_count = 0

# For QA inspection
window_start_tracker = []
window_n_rows_tracker = []
window_min_time_tracker = []
window_max_time_tracker = []

# 3) Build features trip-by-trip
for idx, trip in df_tripdata.iterrows():
    end_time = trip[survey_time_col]
    hrs = trip[hrsf_col]

    if pd.isna(end_time) or pd.isna(hrs) or hrs < 0:
        # Conservative defaults for malformed rows
        df_tripdata.at[idx, "avg_wind_speed"] = 0.0
        df_tripdata.at[idx, "avg_wind_gust"] = 0.0
        df_tripdata.at[idx, "wind_speed_rose"] = 0
        df_tripdata.at[idx, "wind_speed_fell"] = 0
        window_start_tracker.append(pd.NaT)
        window_n_rows_tracker.append(0)
        window_min_time_tracker.append(pd.NaT)
        window_max_time_tracker.append(pd.NaT)
        continue

    start_time = end_time - pd.to_timedelta(hrs, unit="h")
    in_window_mask = (weather_times > start_time) & (weather_times <= end_time)
    trip_weather = weather_work.loc[in_window_mask]

    window_start_tracker.append(start_time)
    window_n_rows_tracker.append(int(trip_weather.shape[0]))
    if trip_weather.empty:
        window_min_time_tracker.append(pd.NaT)
        window_max_time_tracker.append(pd.NaT)
    else:
        window_min_time_tracker.append(trip_weather[weather_time_col].min())
        window_max_time_tracker.append(trip_weather[weather_time_col].max())

    trip_used_fallback = False

    if trip_weather.empty:
        empty_window_count += 1
        trip_used_fallback = True

        # Fallback: nearest weather row by absolute time distance
        nearest_idx = (weather_times - end_time).abs().idxmin()
        avg_speed = float(weather_speed.loc[nearest_idx]) if pd.notna(weather_speed.loc[nearest_idx]) else 0.0
        avg_gust = float(weather_gust.loc[nearest_idx]) if pd.notna(weather_gust.loc[nearest_idx]) else 0.0
        rose = 0
        fell = 0
    else:
        speed_in_window = pd.to_numeric(trip_weather[wind_speed_col], errors="coerce").dropna()
        gust_in_window = pd.to_numeric(trip_weather[wind_gust_col], errors="coerce").dropna()

        if speed_in_window.empty:
            nearest_idx = (weather_times - end_time).abs().idxmin()
            avg_speed = float(weather_speed.loc[nearest_idx]) if pd.notna(weather_speed.loc[nearest_idx]) else 0.0
            trip_used_fallback = True
        else:
            avg_speed = float(speed_in_window.mean())

        if gust_in_window.empty:
            nearest_idx = (weather_times - end_time).abs().idxmin()
            avg_gust = float(weather_gust.loc[nearest_idx]) if pd.notna(weather_gust.loc[nearest_idx]) else 0.0
            trip_used_fallback = True
        else:
            avg_gust = float(gust_in_window.mean())

        if len(speed_in_window) < 2:
            rose = 0
            fell = 0
        else:
            diffs = speed_in_window.diff().dropna()
            rose = int((diffs > 0).any())
            fell = int((diffs < 0).any())

    if trip_used_fallback:
        fallback_trip_count += 1

    df_tripdata.at[idx, "avg_wind_speed"] = avg_speed
    df_tripdata.at[idx, "avg_wind_gust"] = avg_gust
    df_tripdata.at[idx, "wind_speed_rose"] = rose
    df_tripdata.at[idx, "wind_speed_fell"] = fell

# 4) QA checks and summary output
qa_nulls = df_tripdata[["avg_wind_speed", "avg_wind_gust", "wind_speed_rose", "wind_speed_fell"]].isna().sum()
qa_flag_values = {
    "wind_speed_rose_unique": sorted(df_tripdata["wind_speed_rose"].dropna().unique().tolist()),
    "wind_speed_fell_unique": sorted(df_tripdata["wind_speed_fell"].dropna().unique().tolist()),
}

survey_unchanged = df_survey_before.equals(df_survey)
survey_missing_new_cols = all(
    c not in df_survey.columns
    for c in ["avg_wind_speed", "avg_wind_gust", "wind_speed_rose", "wind_speed_fell"]
)

qa_summary = pd.DataFrame(
    {
        "metric": [
            "rows_in_df_survey",
            "rows_in_df_tripdata",
            "empty_weather_windows",
            "trips_using_nearest_fallback",
            "survey_unchanged_check",
            "survey_has_no_new_feature_cols",
        ],
        "value": [
            len(df_survey),
            len(df_tripdata),
            empty_window_count,
            fallback_trip_count,
            survey_unchanged,
            survey_missing_new_cols,
        ],
    }
)

qa_samples = df_tripdata.sample(n=min(5, len(df_tripdata)), random_state=42).copy()
qa_samples["trip_window_start"] = [window_start_tracker[i] for i in qa_samples.index]
qa_samples["weather_rows_in_window"] = [window_n_rows_tracker[i] for i in qa_samples.index]
qa_samples["weather_time_min_in_window"] = [window_min_time_tracker[i] for i in qa_samples.index]
qa_samples["weather_time_max_in_window"] = [window_max_time_tracker[i] for i in qa_samples.index]

display(pd.DataFrame({"assumptions": assumptions}))
display(pd.DataFrame({"selected_columns": [survey_time_col, hrsf_col, weather_time_col, wind_speed_col, wind_gust_col]}))
display(qa_summary)
display(pd.DataFrame({"new_feature_null_counts": qa_nulls}))
display(pd.DataFrame(qa_flag_values))
display(
    qa_samples[[
        survey_time_col,
        hrsf_col,
        "trip_window_start",
        "weather_rows_in_window",
        "weather_time_min_in_window",
        "weather_time_max_in_window",
        "avg_wind_speed",
        "avg_wind_gust",
        "wind_speed_rose",
        "wind_speed_fell",
    ]]
)

print("df_tripdata ready with four engineered wind features.")

,assumptions
0,Trip timestamp marks trip end time in df_survey.
1,"Trip window is (end_time - hrsf, end_time], in..."
2,Weather rows may have time gaps; compare avail...
3,If a trip has no in-window rows OR metric valu...
4,If rise/fall cannot be inferred (fewer than 2 ...


,selected_columns
0,SURVEY_TIMESTAMP
1,HRSF
2,timestamp
3,wind_speed_ms
4,wind_gust_ms


,metric,value
0,rows_in_df_survey,70863
1,rows_in_df_tripdata,70863
2,empty_weather_windows,315
3,trips_using_nearest_fallback,3389
4,survey_unchanged_check,True
5,survey_has_no_new_feature_cols,True


,new_feature_null_counts
avg_wind_speed,0
avg_wind_gust,0
wind_speed_rose,0
wind_speed_fell,0


,wind_speed_rose_unique,wind_speed_fell_unique
0,0,0
1,1,1


,SURVEY_TIMESTAMP,HRSF,trip_window_start,weather_rows_in_window,weather_time_min_in_window,weather_time_max_in_window,avg_wind_speed,avg_wind_gust,wind_speed_rose,wind_speed_fell
58999,2023-10-31 19:00:00,2.0,2023-10-31 17:00:00,2,2023-10-31 18:00:00,2023-10-31 19:00:00,4.500000,6.250000,0,1
53125,2024-11-11 14:00:00,3.0,2024-11-11 11:00:00,3,2024-11-11 12:00:00,2024-11-11 14:00:00,8.133333,9.916667,0,1
56493,2022-07-23 07:00:00,2.0,2022-07-23 05:00:00,2,2022-07-23 06:00:00,2022-07-23 07:00:00,4.616667,5.583333,0,1
50186,2021-06-26 11:00:00,2.5,2021-06-26 08:30:00,3,2021-06-26 09:00:00,2021-06-26 11:00:00,4.155556,5.055556,0,1
59626,2023-04-12 13:00:00,4.0,2023-04-12 09:00:00,4,2023-04-12 10:00:00,2023-04-12 13:00:00,6.750000,7.950000,0,1


df_tripdata ready with four engineered wind features.


In [7]:
df_tripdata.head()

,ID_CODE,SURVEY_TIMESTAMP,FFDAYS2,HRSF,KOD,MODE_F,CNTRBTRS,IMP_REC,ATLANTIC COD,ATLANTIC CROAKER,...,TAUTOG,UNIDENTIFIED (SHARKS),UNIDENTIFIED SKATE OR RAY,WEAKFISH,WINDOWPANE,WINTER FLOUNDER,avg_wind_speed,avg_wind_gust,wind_speed_rose,wind_speed_fell
0,1000920140505017,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6.225000,7.441667,1,1
1,1000920140505018,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6.225000,7.441667,1,1
2,1000920150620006,2015-06-20 16:00:00,0.0,4.5,we,8.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,5.946667,7.013333,0,1
3,1000920150627020,2015-06-27 17:00:00,3.0,7.0,we,8.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,7.223810,8.509524,1,1
4,1000920150711002,2015-07-11 11:00:00,1.0,2.0,we,8.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.566667,4.250000,0,1


In [8]:
# Air-pressure trip features using the same rules as wind features
pressure_col = pick_column(
    weather_work,
    candidates=["air_pressure_hpa", "AIR_PRESSURE_HPA", "air_pressure", "pressure_hpa"],
    df_name="df_weather",
    purpose="air pressure",
)

# Ensure pressure series is numeric
weather_pressure = pd.to_numeric(weather_work[pressure_col], errors="coerce")

# Pre-create columns on df_tripdata
df_tripdata["avg_air_pressure"] = 0.0
df_tripdata["air_pressure_rose"] = 0
df_tripdata["air_pressure_fell"] = 0

pressure_empty_window_count = 0
pressure_fallback_trip_count = 0

# Optional QA trackers for this block
pressure_window_start_tracker = []
pressure_window_n_rows_tracker = []
pressure_window_min_time_tracker = []
pressure_window_max_time_tracker = []

for idx, trip in df_tripdata.iterrows():
    end_time = trip[survey_time_col]
    hrs = trip[hrsf_col]

    if pd.isna(end_time) or pd.isna(hrs) or hrs < 0:
        df_tripdata.at[idx, "avg_air_pressure"] = 0.0
        df_tripdata.at[idx, "air_pressure_rose"] = 0
        df_tripdata.at[idx, "air_pressure_fell"] = 0
        pressure_window_start_tracker.append(pd.NaT)
        pressure_window_n_rows_tracker.append(0)
        pressure_window_min_time_tracker.append(pd.NaT)
        pressure_window_max_time_tracker.append(pd.NaT)
        continue

    start_time = end_time - pd.to_timedelta(hrs, unit="h")
    in_window_mask = (weather_times > start_time) & (weather_times <= end_time)
    trip_weather = weather_work.loc[in_window_mask]

    pressure_window_start_tracker.append(start_time)
    pressure_window_n_rows_tracker.append(int(trip_weather.shape[0]))
    if trip_weather.empty:
        pressure_window_min_time_tracker.append(pd.NaT)
        pressure_window_max_time_tracker.append(pd.NaT)
    else:
        pressure_window_min_time_tracker.append(trip_weather[weather_time_col].min())
        pressure_window_max_time_tracker.append(trip_weather[weather_time_col].max())

    trip_used_fallback = False

    if trip_weather.empty:
        pressure_empty_window_count += 1
        trip_used_fallback = True

        nearest_idx = (weather_times - end_time).abs().idxmin()
        avg_pressure = float(weather_pressure.loc[nearest_idx]) if pd.notna(weather_pressure.loc[nearest_idx]) else 0.0
        pressure_rose = 0
        pressure_fell = 0
    else:
        pressure_in_window = pd.to_numeric(trip_weather[pressure_col], errors="coerce").dropna()

        if pressure_in_window.empty:
            nearest_idx = (weather_times - end_time).abs().idxmin()
            avg_pressure = float(weather_pressure.loc[nearest_idx]) if pd.notna(weather_pressure.loc[nearest_idx]) else 0.0
            trip_used_fallback = True
        else:
            avg_pressure = float(pressure_in_window.mean())

        if len(pressure_in_window) < 2:
            pressure_rose = 0
            pressure_fell = 0
        else:
            pressure_diffs = pressure_in_window.diff().dropna()
            pressure_rose = int((pressure_diffs > 0).any())
            pressure_fell = int((pressure_diffs < 0).any())

    if trip_used_fallback:
        pressure_fallback_trip_count += 1

    df_tripdata.at[idx, "avg_air_pressure"] = avg_pressure
    df_tripdata.at[idx, "air_pressure_rose"] = pressure_rose
    df_tripdata.at[idx, "air_pressure_fell"] = pressure_fell

# QA output for air pressure features
pressure_qa_nulls = df_tripdata[["avg_air_pressure", "air_pressure_rose", "air_pressure_fell"]].isna().sum()
pressure_qa_flags = {
    "air_pressure_rose_unique": sorted(df_tripdata["air_pressure_rose"].dropna().unique().tolist()),
    "air_pressure_fell_unique": sorted(df_tripdata["air_pressure_fell"].dropna().unique().tolist()),
}

pressure_qa_summary = pd.DataFrame(
    {
        "metric": [
            "pressure_empty_weather_windows",
            "pressure_trips_using_nearest_fallback",
        ],
        "value": [
            pressure_empty_window_count,
            pressure_fallback_trip_count,
        ],
    }
)

pressure_qa_samples = df_tripdata.sample(n=min(5, len(df_tripdata)), random_state=123).copy()
pressure_qa_samples["trip_window_start"] = [pressure_window_start_tracker[i] for i in pressure_qa_samples.index]
pressure_qa_samples["weather_rows_in_window"] = [pressure_window_n_rows_tracker[i] for i in pressure_qa_samples.index]
pressure_qa_samples["weather_time_min_in_window"] = [pressure_window_min_time_tracker[i] for i in pressure_qa_samples.index]
pressure_qa_samples["weather_time_max_in_window"] = [pressure_window_max_time_tracker[i] for i in pressure_qa_samples.index]

display(pd.DataFrame({"selected_pressure_column": [pressure_col]}))
display(pressure_qa_summary)
display(pd.DataFrame({"new_pressure_feature_null_counts": pressure_qa_nulls}))
display(pd.DataFrame(pressure_qa_flags))
display(
    pressure_qa_samples[[
        survey_time_col,
        hrsf_col,
        "trip_window_start",
        "weather_rows_in_window",
        "weather_time_min_in_window",
        "weather_time_max_in_window",
        "avg_air_pressure",
        "air_pressure_rose",
        "air_pressure_fell",
    ]]
)

print("df_tripdata now includes avg_air_pressure, air_pressure_rose, and air_pressure_fell.")

,selected_pressure_column
0,air_pressure_hpa


,metric,value
0,pressure_empty_weather_windows,315
1,pressure_trips_using_nearest_fallback,835


,new_pressure_feature_null_counts
avg_air_pressure,0
air_pressure_rose,0
air_pressure_fell,0


,air_pressure_rose_unique,air_pressure_fell_unique
0,0,0
1,1,1


,SURVEY_TIMESTAMP,HRSF,trip_window_start,weather_rows_in_window,weather_time_min_in_window,weather_time_max_in_window,avg_air_pressure,air_pressure_rose,air_pressure_fell
57995,2021-10-03 13:00:00,1.5,2021-10-03 11:30:00,2,2021-10-03 12:00:00,2021-10-03 13:00:00,1015.900000,1,0
38929,2018-05-26 12:00:00,2.5,2018-05-26 09:30:00,3,2018-05-26 10:00:00,2018-05-26 12:00:00,1010.300000,1,0
22331,2016-09-12 17:00:00,0.5,2016-09-12 16:30:00,1,2016-09-12 17:00:00,2016-09-12 17:00:00,1024.166667,0,0
10130,2015-06-26 12:00:00,4.0,2015-06-26 08:00:00,4,2015-06-26 09:00:00,2015-06-26 12:00:00,1013.091667,1,0
1210,2017-04-14 16:00:00,4.0,2017-04-14 12:00:00,4,2017-04-14 13:00:00,2017-04-14 16:00:00,1029.658333,1,1


df_tripdata now includes avg_air_pressure, air_pressure_rose, and air_pressure_fell.


In [9]:
# Trip-window features for air temp, water temp, and water level
metric_specs = [
    {
        "source_col": pick_column(
            weather_work,
            candidates=["air_temp_c", "AIR_TEMP_C", "air_temp"],
            df_name="df_weather",
            purpose="air temperature",
        ),
        "avg_col": "avg_air_temp_c",
        "rose_col": "air_temp_rose",
        "fell_col": "air_temp_fell",
        "label": "air_temp",
    },
    {
        "source_col": pick_column(
            weather_work,
            candidates=["water_temp_c", "WATER_TEMP_C", "water_temp"],
            df_name="df_weather",
            purpose="water temperature",
        ),
        "avg_col": "avg_water_temp_c",
        "rose_col": "water_temp_rose",
        "fell_col": "water_temp_fell",
        "label": "water_temp",
    },
    {
        "source_col": pick_column(
            weather_work,
            candidates=["water_level", "WATER_LEVEL", "water_level_m"],
            df_name="df_weather",
            purpose="water level",
        ),
        "avg_col": "avg_water_level",
        "rose_col": "water_level_rose",
        "fell_col": "water_level_fell",
        "label": "water_level",
    },
]

# Pre-create output columns
for spec in metric_specs:
    df_tripdata[spec["avg_col"]] = 0.0
    df_tripdata[spec["rose_col"]] = 0
    df_tripdata[spec["fell_col"]] = 0

metric_stats = []
sample_trackers = {}

for spec in metric_specs:
    source_col = spec["source_col"]
    avg_col = spec["avg_col"]
    rose_col = spec["rose_col"]
    fell_col = spec["fell_col"]
    label = spec["label"]

    series_all = pd.to_numeric(weather_work[source_col], errors="coerce")

    empty_window_count_metric = 0
    fallback_trip_count_metric = 0

    window_start_tracker_metric = []
    window_n_rows_tracker_metric = []
    window_min_time_tracker_metric = []
    window_max_time_tracker_metric = []

    for idx, trip in df_tripdata.iterrows():
        end_time = trip[survey_time_col]
        hrs = trip[hrsf_col]

        if pd.isna(end_time) or pd.isna(hrs) or hrs < 0:
            df_tripdata.at[idx, avg_col] = 0.0
            df_tripdata.at[idx, rose_col] = 0
            df_tripdata.at[idx, fell_col] = 0
            window_start_tracker_metric.append(pd.NaT)
            window_n_rows_tracker_metric.append(0)
            window_min_time_tracker_metric.append(pd.NaT)
            window_max_time_tracker_metric.append(pd.NaT)
            continue

        start_time = end_time - pd.to_timedelta(hrs, unit="h")
        in_window_mask = (weather_times > start_time) & (weather_times <= end_time)
        trip_weather = weather_work.loc[in_window_mask]

        window_start_tracker_metric.append(start_time)
        window_n_rows_tracker_metric.append(int(trip_weather.shape[0]))
        if trip_weather.empty:
            window_min_time_tracker_metric.append(pd.NaT)
            window_max_time_tracker_metric.append(pd.NaT)
        else:
            window_min_time_tracker_metric.append(trip_weather[weather_time_col].min())
            window_max_time_tracker_metric.append(trip_weather[weather_time_col].max())

        used_fallback = False

        if trip_weather.empty:
            empty_window_count_metric += 1
            used_fallback = True

            nearest_idx = (weather_times - end_time).abs().idxmin()
            avg_value = float(series_all.loc[nearest_idx]) if pd.notna(series_all.loc[nearest_idx]) else 0.0
            rose_value = 0
            fell_value = 0
        else:
            series_window = pd.to_numeric(trip_weather[source_col], errors="coerce").dropna()

            if series_window.empty:
                nearest_idx = (weather_times - end_time).abs().idxmin()
                avg_value = float(series_all.loc[nearest_idx]) if pd.notna(series_all.loc[nearest_idx]) else 0.0
                used_fallback = True
            else:
                avg_value = float(series_window.mean())

            if len(series_window) < 2:
                rose_value = 0
                fell_value = 0
            else:
                diffs = series_window.diff().dropna()
                rose_value = int((diffs > 0).any())
                fell_value = int((diffs < 0).any())

        if used_fallback:
            fallback_trip_count_metric += 1

        df_tripdata.at[idx, avg_col] = avg_value
        df_tripdata.at[idx, rose_col] = rose_value
        df_tripdata.at[idx, fell_col] = fell_value

    metric_stats.append(
        {
            "metric": label,
            "source_column": source_col,
            "empty_weather_windows": empty_window_count_metric,
            "trips_using_nearest_fallback": fallback_trip_count_metric,
            "avg_nulls": int(df_tripdata[avg_col].isna().sum()),
            "rose_nulls": int(df_tripdata[rose_col].isna().sum()),
            "fell_nulls": int(df_tripdata[fell_col].isna().sum()),
            "rose_unique": sorted(df_tripdata[rose_col].dropna().unique().tolist()),
            "fell_unique": sorted(df_tripdata[fell_col].dropna().unique().tolist()),
        }
    )

    sample_trackers[label] = {
        "window_start": window_start_tracker_metric,
        "window_rows": window_n_rows_tracker_metric,
        "window_min": window_min_time_tracker_metric,
        "window_max": window_max_time_tracker_metric,
        "avg_col": avg_col,
        "rose_col": rose_col,
        "fell_col": fell_col,
    }

# QA summary tables
metric_stats_df = pd.DataFrame(metric_stats)
display(metric_stats_df[["metric", "source_column", "empty_weather_windows", "trips_using_nearest_fallback"]])
display(
    metric_stats_df[[
        "metric",
        "avg_nulls",
        "rose_nulls",
        "fell_nulls",
        "rose_unique",
        "fell_unique",
    ]]
)

# Show random samples for each metric
for label, tracker in sample_trackers.items():
    sample = df_tripdata.sample(n=min(5, len(df_tripdata)), random_state=77).copy()
    sample["trip_window_start"] = [tracker["window_start"][i] for i in sample.index]
    sample["weather_rows_in_window"] = [tracker["window_rows"][i] for i in sample.index]
    sample["weather_time_min_in_window"] = [tracker["window_min"][i] for i in sample.index]
    sample["weather_time_max_in_window"] = [tracker["window_max"][i] for i in sample.index]

    display(
        sample[[
            survey_time_col,
            hrsf_col,
            "trip_window_start",
            "weather_rows_in_window",
            "weather_time_min_in_window",
            "weather_time_max_in_window",
            tracker["avg_col"],
            tracker["rose_col"],
            tracker["fell_col"],
        ]].head()
    )

print(
    "df_tripdata now includes avg/rose/fell for air temp, water temp, and water level.",
    "Columns added:",
    "avg_air_temp_c, air_temp_rose, air_temp_fell,",
    "avg_water_temp_c, water_temp_rose, water_temp_fell,",
    "avg_water_level, water_level_rose, water_level_fell",
)

,metric,source_column,empty_weather_windows,trips_using_nearest_fallback
0,air_temp,air_temp_c,315,10024
1,water_temp,water_temp_c,315,1707
2,water_level,water_level,315,315


,metric,avg_nulls,rose_nulls,fell_nulls,rose_unique,fell_unique
0,air_temp,0,0,0,"[0, 1]","[0, 1]"
1,water_temp,0,0,0,"[0, 1]","[0, 1]"
2,water_level,0,0,0,"[0, 1]","[0, 1]"


,SURVEY_TIMESTAMP,HRSF,trip_window_start,weather_rows_in_window,weather_time_min_in_window,weather_time_max_in_window,avg_air_temp_c,air_temp_rose,air_temp_fell
8364,2015-11-15 15:00:00,6.0,2015-11-15 09:00:00,6,2015-11-15 10:00:00,2015-11-15 15:00:00,8.20,1,1
41323,2019-09-16 19:00:00,1.0,2019-09-16 18:00:00,1,2019-09-16 19:00:00,2019-09-16 19:00:00,21.40,0,0
9897,2014-09-06 15:00:00,5.0,2014-09-06 10:00:00,5,2014-09-06 11:00:00,2014-09-06 15:00:00,0.00,0,0
153,2014-04-14 16:00:00,5.0,2014-04-14 11:00:00,5,2014-04-14 12:00:00,2014-04-14 16:00:00,9.61,1,1
45919,2020-09-06 12:00:00,6.0,2020-09-06 06:00:00,6,2020-09-06 07:00:00,2020-09-06 12:00:00,21.20,1,1


,SURVEY_TIMESTAMP,HRSF,trip_window_start,weather_rows_in_window,weather_time_min_in_window,weather_time_max_in_window,avg_water_temp_c,water_temp_rose,water_temp_fell
8364,2015-11-15 15:00:00,6.0,2015-11-15 09:00:00,6,2015-11-15 10:00:00,2015-11-15 15:00:00,15.072222,1,0
41323,2019-09-16 19:00:00,1.0,2019-09-16 18:00:00,1,2019-09-16 19:00:00,2019-09-16 19:00:00,22.400000,0,0
9897,2014-09-06 15:00:00,5.0,2014-09-06 10:00:00,5,2014-09-06 11:00:00,2014-09-06 15:00:00,0.000000,0,0
153,2014-04-14 16:00:00,5.0,2014-04-14 11:00:00,5,2014-04-14 12:00:00,2014-04-14 16:00:00,6.300000,1,1
45919,2020-09-06 12:00:00,6.0,2020-09-06 06:00:00,6,2020-09-06 07:00:00,2020-09-06 12:00:00,22.827778,1,1


,SURVEY_TIMESTAMP,HRSF,trip_window_start,weather_rows_in_window,weather_time_min_in_window,weather_time_max_in_window,avg_water_level,water_level_rose,water_level_fell
8364,2015-11-15 15:00:00,6.0,2015-11-15 09:00:00,6,2015-11-15 10:00:00,2015-11-15 15:00:00,0.314667,1,1
41323,2019-09-16 19:00:00,1.0,2019-09-16 18:00:00,1,2019-09-16 19:00:00,2019-09-16 19:00:00,2.371000,0,0
9897,2014-09-06 15:00:00,5.0,2014-09-06 10:00:00,5,2014-09-06 11:00:00,2014-09-06 15:00:00,2.258200,1,1
153,2014-04-14 16:00:00,5.0,2014-04-14 11:00:00,5,2014-04-14 12:00:00,2014-04-14 16:00:00,1.667800,1,0
45919,2020-09-06 12:00:00,6.0,2020-09-06 06:00:00,6,2020-09-06 07:00:00,2020-09-06 12:00:00,1.303333,0,1


df_tripdata now includes avg/rose/fell for air temp, water temp, and water level. Columns added: avg_air_temp_c, air_temp_rose, air_temp_fell, avg_water_temp_c, water_temp_rose, water_temp_fell, avg_water_level, water_level_rose, water_level_fell


In [10]:
# Comprehensive QA checks for engineered trip features
import numpy as np
import pandas as pd

qa_required_avg_cols = [
    "avg_wind_speed",
    "avg_wind_gust",
    "avg_air_pressure",
    "avg_air_temp_c",
    "avg_water_temp_c",
    "avg_water_level",
]
qa_required_flag_cols = [
    "wind_speed_rose", "wind_speed_fell",
    "air_pressure_rose", "air_pressure_fell",
    "air_temp_rose", "air_temp_fell",
    "water_temp_rose", "water_temp_fell",
    "water_level_rose", "water_level_fell",
]
qa_required_all = qa_required_avg_cols + qa_required_flag_cols

# 1) Structural checks
missing_feature_cols = [c for c in qa_required_all if c not in df_tripdata.columns]
row_count_ok = len(df_tripdata) == len(df_survey)

# 2) Null and flag-value checks
null_counts = df_tripdata[qa_required_all].isna().sum() if not missing_feature_cols else pd.Series(dtype=int)
flag_value_checks = {}
for col in qa_required_flag_cols:
    if col in df_tripdata.columns:
        uniq = sorted(pd.Series(df_tripdata[col]).dropna().unique().tolist())
        flag_value_checks[col] = {"unique_values": uniq, "is_binary": set(uniq).issubset({0, 1})}

# 3) Optional sanity checks for numeric columns
avg_col_stats = []
for col in qa_required_avg_cols:
    if col in df_tripdata.columns:
        s = pd.to_numeric(df_tripdata[col], errors="coerce")
        avg_col_stats.append(
            {
                "column": col,
                "min": float(s.min()) if s.notna().any() else np.nan,
                "max": float(s.max()) if s.notna().any() else np.nan,
                "mean": float(s.mean()) if s.notna().any() else np.nan,
                "zeros": int((s == 0).sum()),
            }
        )
avg_col_stats_df = pd.DataFrame(avg_col_stats)

# 4) Logic spot-check by recomputing sample rows
sample_n = min(50, len(df_tripdata))
sample_idx = df_tripdata.sample(n=sample_n, random_state=2026).index
sample_rows = df_tripdata.loc[sample_idx]

recompute_specs = [
    {"source": wind_speed_col, "avg": "avg_wind_speed", "rose": "wind_speed_rose", "fell": "wind_speed_fell"},
    {"source": wind_gust_col, "avg": "avg_wind_gust", "rose": None, "fell": None},
    {"source": pressure_col, "avg": "avg_air_pressure", "rose": "air_pressure_rose", "fell": "air_pressure_fell"},
    {"source": "air_temp_c", "avg": "avg_air_temp_c", "rose": "air_temp_rose", "fell": "air_temp_fell"},
    {"source": "water_temp_c", "avg": "avg_water_temp_c", "rose": "water_temp_rose", "fell": "water_temp_fell"},
    {"source": "water_level", "avg": "avg_water_level", "rose": "water_level_rose", "fell": "water_level_fell"},
]

def recompute_trip_metric(end_time, hrs, source_col, include_flags=True):
    if pd.isna(end_time) or pd.isna(hrs) or hrs < 0:
        if include_flags:
            return 0.0, 0, 0
        return 0.0, None, None

    start_time = end_time - pd.to_timedelta(hrs, unit="h")
    in_window = weather_work.loc[(weather_times > start_time) & (weather_times <= end_time)]
    series_all_local = pd.to_numeric(weather_work[source_col], errors="coerce")

    if in_window.empty:
        nearest_idx = (weather_times - end_time).abs().idxmin()
        avg_val = float(series_all_local.loc[nearest_idx]) if pd.notna(series_all_local.loc[nearest_idx]) else 0.0
        if include_flags:
            return avg_val, 0, 0
        return avg_val, None, None

    series_window = pd.to_numeric(in_window[source_col], errors="coerce").dropna()
    if series_window.empty:
        nearest_idx = (weather_times - end_time).abs().idxmin()
        avg_val = float(series_all_local.loc[nearest_idx]) if pd.notna(series_all_local.loc[nearest_idx]) else 0.0
        if include_flags:
            return avg_val, 0, 0
        return avg_val, None, None

    avg_val = float(series_window.mean())
    if not include_flags:
        return avg_val, None, None

    if len(series_window) < 2:
        return avg_val, 0, 0

    diffs = series_window.diff().dropna()
    return avg_val, int((diffs > 0).any()), int((diffs < 0).any())

sample_mismatch_rows = []
for idx, row in sample_rows.iterrows():
    end_time = row[survey_time_col]
    hrs = row[hrsf_col]

    for spec in recompute_specs:
        include_flags = spec["rose"] is not None and spec["fell"] is not None
        recomputed_avg, recomputed_rose, recomputed_fell = recompute_trip_metric(
            end_time=end_time,
            hrs=hrs,
            source_col=spec["source"],
            include_flags=include_flags,
        )

        actual_avg = float(row[spec["avg"]])
        if not np.isclose(actual_avg, recomputed_avg, atol=1e-9, rtol=0):
            sample_mismatch_rows.append(
                {
                    "index": int(idx),
                    "feature": spec["avg"],
                    "actual": actual_avg,
                    "recomputed": recomputed_avg,
                }
            )

        if include_flags:
            actual_rose = int(row[spec["rose"]])
            actual_fell = int(row[spec["fell"]])
            if actual_rose != recomputed_rose:
                sample_mismatch_rows.append(
                    {
                        "index": int(idx),
                        "feature": spec["rose"],
                        "actual": actual_rose,
                        "recomputed": recomputed_rose,
                    }
                )
            if actual_fell != recomputed_fell:
                sample_mismatch_rows.append(
                    {
                        "index": int(idx),
                        "feature": spec["fell"],
                        "actual": actual_fell,
                        "recomputed": recomputed_fell,
                    }
                )

sample_mismatch_df = pd.DataFrame(sample_mismatch_rows)

qa_summary_rows = [
    {"check": "missing_feature_columns", "result": len(missing_feature_cols) == 0, "detail": str(missing_feature_cols)},
    {"check": "row_count_matches_df_survey", "result": row_count_ok, "detail": f"df_tripdata={len(df_tripdata)}, df_survey={len(df_survey)}"},
    {"check": "no_nulls_in_engineered_features", "result": bool((null_counts.sum() == 0) if len(null_counts) else False), "detail": str(null_counts.to_dict())},
    {"check": "all_flag_columns_binary", "result": all(v["is_binary"] for v in flag_value_checks.values()) if flag_value_checks else False, "detail": str({k: v['unique_values'] for k, v in flag_value_checks.items()})},
    {"check": "sample_recompute_matches", "result": sample_mismatch_df.empty, "detail": f"sample_size={sample_n}, mismatches={len(sample_mismatch_df)}"},
]
qa_summary_df = pd.DataFrame(qa_summary_rows)

display(qa_summary_df)
display(avg_col_stats_df)
if not sample_mismatch_df.empty:
    display(sample_mismatch_df.head(30))

if qa_summary_df["result"].all():
    print("QA PASS: Engineered trip features passed structural, null, binary-flag, and sample recomputation checks.")
else:
    print("QA WARNING: One or more QA checks failed. Review the tables above for details.")

,check,result,detail
0,missing_feature_columns,True,[]
1,row_count_matches_df_survey,True,"df_tripdata=70863, df_survey=70863"
2,no_nulls_in_engineered_features,True,"{'avg_wind_speed': 0, 'avg_wind_gust': 0, 'avg..."
3,all_flag_columns_binary,True,"{'wind_speed_rose': [0, 1], 'wind_speed_fell':..."
4,sample_recompute_matches,True,"sample_size=50, mismatches=0"


,column,min,max,mean,zeros
0,avg_wind_speed,0.000,17.811111,4.740825,3166
1,avg_wind_gust,0.000,22.488889,5.913543,3162
2,avg_air_pressure,0.000,1048.160000,1009.425545,555
3,avg_air_temp_c,-12.220,30.550000,15.059248,9880
4,avg_water_temp_c,0.000,27.450000,17.963696,1474
5,avg_water_level,-0.616,3.315000,1.226841,5


QA PASS: Engineered trip features passed structural, null, binary-flag, and sample recomputation checks.


In [13]:
# Additional trip features: sun/moon event flags, moon phase age, and distances (dewpoint removed)
import numpy as np
import pandas as pd

def _resolve_col(df: pd.DataFrame, candidates: list[str], purpose: str) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    raise ValueError(f"Could not find column for {purpose}. Tried {candidates}")

# Remove dewpoint feature entirely if present from previous runs
if "avg_dewpoint" in df_tripdata.columns:
    df_tripdata = df_tripdata.drop(columns=["avg_dewpoint"])

# Resolve source columns
sunrise_col = _resolve_col(weather_work, ["sunrise_time", "SUNRISE_TIME"], "sunrise time")
sunset_col = _resolve_col(weather_work, ["sunset_time", "SUNSET_TIME"], "sunset time")
moonrise_col = _resolve_col(weather_work, ["moonrise_time", "MOONRISE_TIME"], "moonrise time")
moonset_col = _resolve_col(weather_work, ["moonset_time", "MOONSET_TIME"], "moonset time")
moon_phase_col = _resolve_col(weather_work, ["moon_phase_age_days", "MOON_PHASE_AGE_DAYS"], "moon phase age")
sun_distance_col = _resolve_col(weather_work, ["sun_distance_km", "SUN_DISTANCE_KM"], "sun distance")
moon_distance_col = _resolve_col(weather_work, ["moon_distance_km", "MOON_DISTANCE_KM"], "moon distance")

# Parse event timestamp columns once
sunrise_series = pd.to_datetime(weather_work[sunrise_col], errors="coerce")
sunset_series = pd.to_datetime(weather_work[sunset_col], errors="coerce")
moonrise_series = pd.to_datetime(weather_work[moonrise_col], errors="coerce")
moonset_series = pd.to_datetime(weather_work[moonset_col], errors="coerce")

# Numeric source series for fallback
moon_phase_series = pd.to_numeric(weather_work[moon_phase_col], errors="coerce")
sun_distance_series = pd.to_numeric(weather_work[sun_distance_col], errors="coerce")
moon_distance_series = pd.to_numeric(weather_work[moon_distance_col], errors="coerce")

# Create output columns on df_tripdata
new_cols = [
    "was_sunrise",
    "was_sunset",
    "was_moonrise",
    "was_moonset",
    "moon_phase_age",
    "sun_distance",
    "moon_distance",
]
for c in new_cols:
    if c.startswith("was_"):
        df_tripdata[c] = 0
    else:
        df_tripdata[c] = 0.0

# Counters for QA
empty_window_count_extra = 0
fallback_moon_phase_count = 0
fallback_sun_distance_count = 0
fallback_moon_distance_count = 0

# Track sample window stats
extra_window_start_tracker = []
extra_window_rows_tracker = []

for idx, trip in df_tripdata.iterrows():
    end_time = trip[survey_time_col]
    hrs = trip[hrsf_col]

    if pd.isna(end_time) or pd.isna(hrs) or hrs < 0:
        extra_window_start_tracker.append(pd.NaT)
        extra_window_rows_tracker.append(0)
        continue

    start_time = end_time - pd.to_timedelta(hrs, unit="h")
    in_window_mask = (weather_times > start_time) & (weather_times <= end_time)
    trip_weather = weather_work.loc[in_window_mask]

    extra_window_start_tracker.append(start_time)
    extra_window_rows_tracker.append(int(trip_weather.shape[0]))

    if trip_weather.empty:
        empty_window_count_extra += 1

    # Average metrics with same fallback logic
    def avg_with_fallback(source_series_all: pd.Series, source_col_name: str):
        used_fallback = False
        if trip_weather.empty:
            nearest_idx = (weather_times - end_time).abs().idxmin()
            val = float(source_series_all.loc[nearest_idx]) if pd.notna(source_series_all.loc[nearest_idx]) else 0.0
            return val, True

        series_window = pd.to_numeric(trip_weather[source_col_name], errors="coerce").dropna()
        if series_window.empty:
            nearest_idx = (weather_times - end_time).abs().idxmin()
            val = float(source_series_all.loc[nearest_idx]) if pd.notna(source_series_all.loc[nearest_idx]) else 0.0
            used_fallback = True
            return val, used_fallback

        return float(series_window.mean()), used_fallback

    avg_moon_phase, used_fb = avg_with_fallback(moon_phase_series, moon_phase_col)
    if used_fb:
        fallback_moon_phase_count += 1

    avg_sun_distance, used_fb = avg_with_fallback(sun_distance_series, sun_distance_col)
    if used_fb:
        fallback_sun_distance_count += 1

    avg_moon_distance, used_fb = avg_with_fallback(moon_distance_series, moon_distance_col)
    if used_fb:
        fallback_moon_distance_count += 1

    # Event flags within the trip window
    if trip_weather.empty:
        was_sunrise = 0
        was_sunset = 0
        was_moonrise = 0
        was_moonset = 0
    else:
        sunrise_in_window = sunrise_series.loc[trip_weather.index].dropna()
        sunset_in_window = sunset_series.loc[trip_weather.index].dropna()
        moonrise_in_window = moonrise_series.loc[trip_weather.index].dropna()
        moonset_in_window = moonset_series.loc[trip_weather.index].dropna()

        was_sunrise = int(((sunrise_in_window > start_time) & (sunrise_in_window <= end_time)).any())
        was_sunset = int(((sunset_in_window > start_time) & (sunset_in_window <= end_time)).any())
        was_moonrise = int(((moonrise_in_window > start_time) & (moonrise_in_window <= end_time)).any())
        was_moonset = int(((moonset_in_window > start_time) & (moonset_in_window <= end_time)).any())

    df_tripdata.at[idx, "was_sunrise"] = was_sunrise
    df_tripdata.at[idx, "was_sunset"] = was_sunset
    df_tripdata.at[idx, "was_moonrise"] = was_moonrise
    df_tripdata.at[idx, "was_moonset"] = was_moonset
    df_tripdata.at[idx, "moon_phase_age"] = avg_moon_phase
    df_tripdata.at[idx, "sun_distance"] = avg_sun_distance
    df_tripdata.at[idx, "moon_distance"] = avg_moon_distance

# QA outputs for this feature block
new_null_counts = df_tripdata[new_cols].isna().sum()
new_flag_uniques = {
    "was_sunrise_unique": sorted(df_tripdata["was_sunrise"].dropna().unique().tolist()),
    "was_sunset_unique": sorted(df_tripdata["was_sunset"].dropna().unique().tolist()),
    "was_moonrise_unique": sorted(df_tripdata["was_moonrise"].dropna().unique().tolist()),
    "was_moonset_unique": sorted(df_tripdata["was_moonset"].dropna().unique().tolist()),
}

new_block_summary = pd.DataFrame(
    {
        "metric": [
            "empty_weather_windows",
            "fallback_moon_phase_age",
            "fallback_sun_distance",
            "fallback_moon_distance",
            "avg_dewpoint_removed",
        ],
        "value": [
            empty_window_count_extra,
            fallback_moon_phase_count,
            fallback_sun_distance_count,
            fallback_moon_distance_count,
            "avg_dewpoint" not in df_tripdata.columns,
        ],
    }
)

sample_extra = df_tripdata.sample(n=min(5, len(df_tripdata)), random_state=404).copy()
sample_extra["trip_window_start"] = [extra_window_start_tracker[i] for i in sample_extra.index]
sample_extra["weather_rows_in_window"] = [extra_window_rows_tracker[i] for i in sample_extra.index]

display(
    pd.DataFrame(
        {"selected_source_columns": [
            sunrise_col, sunset_col, moonrise_col, moonset_col,
            moon_phase_col, sun_distance_col, moon_distance_col
        ]}
    )
)
display(new_block_summary)
display(pd.DataFrame({"new_block_null_counts": new_null_counts}))
display(pd.DataFrame(new_flag_uniques))
display(
    sample_extra[[
        survey_time_col,
        hrsf_col,
        "trip_window_start",
        "weather_rows_in_window",
        "was_sunrise",
        "was_sunset",
        "was_moonrise",
        "was_moonset",
        "moon_phase_age",
        "sun_distance",
        "moon_distance",
    ]]
)

print("df_tripdata updated: dewpoint feature removed; sun/moon and astronomy features retained.")

,selected_source_columns
0,sunrise_time
1,sunset_time
2,moonrise_time
3,moonset_time
4,moon_phase_age_days
5,sun_distance_km
6,moon_distance_km


,metric,value
0,empty_weather_windows,315
1,fallback_moon_phase_age,315
2,fallback_sun_distance,315
3,fallback_moon_distance,315
4,avg_dewpoint_removed,True


,new_block_null_counts
was_sunrise,0
was_sunset,0
was_moonrise,0
was_moonset,0
moon_phase_age,0
sun_distance,0
moon_distance,0


,was_sunrise_unique,was_sunset_unique,was_moonrise_unique,was_moonset_unique
0,0,0,0,0
1,1,1,1,1


,SURVEY_TIMESTAMP,HRSF,trip_window_start,weather_rows_in_window,was_sunrise,was_sunset,was_moonrise,was_moonset,moon_phase_age,sun_distance,moon_distance
51185,2023-04-07 13:00:00,5.0,2023-04-07 08:00:00,5,0,0,0,0,15.200111,1.497207e+08,386433.350502
39151,2018-08-10 13:00:00,2.0,2018-08-10 11:00:00,2,0,0,0,0,26.866778,1.516454e+08,354218.587680
2436,2020-12-04 17:00:00,6.0,2020-12-04 11:00:00,6,0,1,0,0,17.611222,1.474268e+08,387763.498884
30178,2017-10-07 14:00:00,6.0,2017-10-07 08:00:00,6,0,0,0,1,15.744556,1.494973e+08,367039.716033
34869,2018-07-08 15:00:00,3.0,2018-07-08 12:00:00,3,0,0,0,0,23.055667,1.520942e+08,370316.204864


df_tripdata updated: dewpoint feature removed; sun/moon and astronomy features retained.


In [14]:
df_tripdata.head()

,ID_CODE,SURVEY_TIMESTAMP,FFDAYS2,HRSF,KOD,MODE_F,CNTRBTRS,IMP_REC,ATLANTIC COD,ATLANTIC CROAKER,...,avg_water_level,water_level_rose,water_level_fell,was_sunrise,was_sunset,was_moonrise,was_moonset,moon_phase_age,sun_distance,moon_distance
0,1000920140505017,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.491667,1,1,0,0,1,0,5.633444,1.508852e+08,404508.533199
1,1000920140505018,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.491667,1,1,0,0,1,0,5.633444,1.508852e+08,404508.533199
2,1000920150620006,2015-06-20 16:00:00,0.0,4.5,we,8.0,1.0,0.0,0.0,0.0,...,0.442800,1,1,0,0,0,0,3.611222,1.520199e+08,397917.383118
3,1000920150627020,2015-06-27 17:00:00,3.0,7.0,we,8.0,0.0,0.0,0.0,0.0,...,1.628286,1,1,0,0,1,0,9.600111,1.520708e+08,400109.337422
4,1000920150711002,2015-07-11 11:00:00,1.0,2.0,we,8.0,1.0,0.0,0.0,0.0,...,2.077000,1,0,0,0,0,0,23.600111,1.520861e+08,372690.223940


In [15]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87598 entries, 0 to 87597
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            87598 non-null  datetime64[ns]
 1   water_level          87598 non-null  float64       
 2   wind_speed_ms        84713 non-null  float64       
 3   wind_gust_ms         84713 non-null  float64       
 4   air_temp_c           76404 non-null  float64       
 5   water_temp_c         83606 non-null  float64       
 6   dewpoint_c           73891 non-null  float64       
 7   wind_direction_deg   76542 non-null  float64       
 8   air_pressure_hpa     86887 non-null  float64       
 9   sunrise_time         87598 non-null  object        
 10  sunset_time          87598 non-null  object        
 11  moonrise_time        81473 non-null  object        
 12  moonset_time         81494 non-null  object        
 13  moon_phase_age_days  87598 non-

In [16]:
df_tripdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70863 entries, 0 to 70862
Data columns (total 66 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   ID_CODE                    70863 non-null  int64         
 1   SURVEY_TIMESTAMP           70863 non-null  datetime64[ns]
 2   FFDAYS2                    70863 non-null  float64       
 3   HRSF                       70863 non-null  float64       
 4   KOD                        70862 non-null  object        
 5   MODE_F                     70863 non-null  float64       
 6   CNTRBTRS                   70863 non-null  float64       
 7   IMP_REC                    70863 non-null  float64       
 8   ATLANTIC COD               70863 non-null  float64       
 9   ATLANTIC CROAKER           70863 non-null  float64       
 10  ATLANTIC HERRING           70863 non-null  float64       
 11  ATLANTIC MACKEREL          70863 non-null  float64       
 12  ATLA

In [19]:
df_tripdata = df_tripdata[df_tripdata["KOD"].notna()].reset_index(drop=True)

In [22]:
df_tripdata.to_csv("../data/processed/df_tripdata_engineered.csv", index=False)

In [23]:
df_sanity_check = pd.read_csv('../data/processed/df_tripdata_engineered.csv')
df_sanity_check.head()

,ID_CODE,SURVEY_TIMESTAMP,FFDAYS2,HRSF,KOD,MODE_F,CNTRBTRS,IMP_REC,ATLANTIC COD,ATLANTIC CROAKER,...,avg_water_level,water_level_rose,water_level_fell,was_sunrise,was_sunset,was_moonrise,was_moonset,moon_phase_age,sun_distance,moon_distance
0,1000920140505017,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.491667,1,1,0,0,1,0,5.633444,1.508852e+08,404508.533199
1,1000920140505018,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.491667,1,1,0,0,1,0,5.633444,1.508852e+08,404508.533199
2,1000920150620006,2015-06-20 16:00:00,0.0,4.5,we,8.0,1.0,0.0,0.0,0.0,...,0.442800,1,1,0,0,0,0,3.611222,1.520199e+08,397917.383118
3,1000920150627020,2015-06-27 17:00:00,3.0,7.0,we,8.0,0.0,0.0,0.0,0.0,...,1.628286,1,1,0,0,1,0,9.600111,1.520708e+08,400109.337422
4,1000920150711002,2015-07-11 11:00:00,1.0,2.0,we,8.0,1.0,0.0,0.0,0.0,...,2.077000,1,0,0,0,0,0,23.600111,1.520861e+08,372690.223940
